# GRIDPILOT AI — Member 2 — Task 4
## Demand Forecasting Baselines

**Purpose:** Implement and evaluate persistence and seasonal-naive baselines on a chronological train/val/test split, producing the MAE/RMSE/MAPE bar that LightGBM must beat.


## 0. Setup

In [ ]:
!pip -q install pandas numpy scikit-learn pyarrow

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Libraries loaded.")

## 1. Load feature-engineered dataset

In [ ]:
FEATURES_PATH = "./feature_engineering_outputs/demand_features.parquet"

df = pd.read_parquet(FEATURES_PATH)
df = df.sort_values("timestamp").reset_index(drop=True)
DEMAND_COL = "demand" if "demand" in df.columns else df.columns[1]
print(df.shape)

## 2. Chronological train / validation / test split

Never shuffle time-series data.

In [ ]:
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print("Train:", train_df["timestamp"].min(), "->", train_df["timestamp"].max(), len(train_df))
print("Val:  ", val_df["timestamp"].min(), "->", val_df["timestamp"].max(), len(val_df))
print("Test: ", test_df["timestamp"].min(), "->", test_df["timestamp"].max(), len(test_df))

## 3. Baseline 1 — Persistence (last-value)

Prediction at t+h = observed value at t.

In [ ]:
FREQ_MINUTES = None  # set from Task 1 findings
assert FREQ_MINUTES is not None, "Set FREQ_MINUTES before running."

HORIZONS_MINUTES = [15, 30, 60]

def persistence_forecast(data, target_col, horizon_minutes, freq_minutes):
    periods = int(horizon_minutes / freq_minutes)
    return data[target_col].shift(periods)

persistence_preds = {}
for h in HORIZONS_MINUTES:
    persistence_preds[h] = persistence_forecast(test_df, DEMAND_COL, h, FREQ_MINUTES)

persistence_preds[HORIZONS_MINUTES[0]].head()

## 4. Baseline 2 — Seasonal naive

Prediction at t+h = observed value at the same time on the previous seasonal cycle (e.g. same time yesterday).

In [ ]:
SEASONAL_CYCLE_MINUTES = 24 * 60  # daily seasonality; adjust if weekly seasonality is more appropriate

def seasonal_naive_forecast(data, target_col, cycle_minutes, freq_minutes):
    periods = int(cycle_minutes / freq_minutes)
    return data[target_col].shift(periods)

seasonal_pred = seasonal_naive_forecast(test_df, DEMAND_COL, SEASONAL_CYCLE_MINUTES, FREQ_MINUTES)
seasonal_pred.head()

## 5. Evaluation metrics

In [ ]:
def evaluate(y_true, y_pred):
    mask = y_true.notna() & y_pred.notna()
    y_true_m, y_pred_m = y_true[mask], y_pred[mask]
    if len(y_true_m) == 0:
        return {"mae": None, "rmse": None, "mape": None, "n": 0}
    mae = mean_absolute_error(y_true_m, y_pred_m)
    rmse = mean_squared_error(y_true_m, y_pred_m, squared=False)
    nonzero = y_true_m != 0
    mape = (np.abs((y_true_m[nonzero] - y_pred_m[nonzero]) / y_true_m[nonzero])).mean() * 100 if nonzero.any() else None
    return {"mae": mae, "rmse": rmse, "mape": mape, "n": int(mask.sum())}

results = []

for h in HORIZONS_MINUTES:
    metrics = evaluate(test_df[DEMAND_COL], persistence_preds[h])
    results.append({"model": "persistence", "horizon_minutes": h, **metrics})

seasonal_metrics = evaluate(test_df[DEMAND_COL], seasonal_pred)
results.append({"model": "seasonal_naive", "horizon_minutes": SEASONAL_CYCLE_MINUTES, **seasonal_metrics})

results_df = pd.DataFrame(results)
display(results_df)

## 6. Save evaluation artifact

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path("./baseline_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

results_df.to_csv(OUTPUT_DIR / "baseline_evaluation.csv", index=False)

with open(OUTPUT_DIR / "baseline_evaluation.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved baseline evaluation artifacts to", OUTPUT_DIR)

## Next step

These numbers are the bar the LightGBM model (Chunk 5) must clear. Do not claim the ML model is better unless it measurably beats these baselines on the same test set.